In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

In [2]:
path = '/media/dff/Elements1/ITS_semester3/Datamining/Association Rule/Datas/online+retail/dt.xlsx'
df = pd.read_excel(path)

# EDA

In [4]:
display(df.head())
print("Ukuran dataset:", df.shape)
display(df.info())
display(df.describe())
display(df.duplicated().sum())
display(df.isna().sum())



,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


Ukuran dataset: (541909, 8)
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


None

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


np.int64(5268)

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [4]:
display(df[df['InvoiceNo'].map(df['InvoiceNo'].value_counts(dropna=False))>1])
display(df[df['InvoiceNo'].map(df['InvoiceNo'].value_counts(dropna=False))==1])

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
25,536369,21756,BATH BUILDING BLOCK WORD,3,2010-12-01 08:35:00,5.95,13047.0,United Kingdom
46,536371,22086,PAPER CHAIN KIT 50'S CHRISTMAS,80,2010-12-01 09:00:00,2.55,13748.0,United Kingdom
65,536374,21258,VICTORIAN SEWING BOX LARGE,32,2010-12-01 09:09:00,10.95,15100.0,United Kingdom
105,536380,22961,JAM MAKING SET PRINTED,24,2010-12-01 09:41:00,1.45,17809.0,United Kingdom
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
...,...,...,...,...,...,...,...,...
540442,581487,21137,BLACK RECORD COVER FRAME,120,2011-12-09 09:44:00,3.39,15694.0,United Kingdom
540450,581491,23571,TRADITIONAL NAUGHTS & CROSSES,12,2011-12-09 10:02:00,1.65,12433.0,Norway
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
541702,581566,23404,HOME SWEET HOME BLACKBOARD,144,2011-12-09 11:50:00,3.26,18102.0,United Kingdom


In [7]:
# 1. Aggregate unique counts per InvoiceNo
invoice_check = df.groupby('InvoiceNo').agg(
    date_count=('InvoiceDate', 'nunique'),
    customer_count=('CustomerID', lambda x: x.nunique(dropna=False))
)

# 2. Filter for InvoiceNos with more than 1 unique date OR customer ID
mismatched_invoices = invoice_check[
    (invoice_check['date_count'] > 1) | (invoice_check['customer_count'] > 1)
]

print(f"Total mismatched InvoiceNos: {len(mismatched_invoices)}")
display(mismatched_invoices)

# 3. Display full records (Cast InvoiceNo to str & CustomerID to Int64)
mismatched_df = df[df['InvoiceNo'].isin(mismatched_invoices.index)].copy()
mismatched_df['InvoiceNo'] = mismatched_df['InvoiceNo'].astype(str)
mismatched_df['CustomerID'] = mismatched_df['CustomerID'].astype('Int64')

display(mismatched_df.sort_values(by='InvoiceNo'))

# 4. Filter and aggregate time differences
date_diffs = (
    df.groupby('InvoiceNo')
    .filter(lambda x: x['InvoiceDate'].nunique() > 1)
    .groupby('InvoiceNo')
    .agg(
        min_date=('InvoiceDate', 'min'),
        max_date=('InvoiceDate', 'max'),
        time_diff=('InvoiceDate', lambda x: x.max() - x.min()),
        unique_dates=('InvoiceDate', 'nunique'),
        customer_ids=('CustomerID', lambda x: list(x.dropna().astype('Int64').unique()))
    )
    .sort_values(by='time_diff', ascending=False)
)

display(date_diffs)

Total mismatched InvoiceNos: 43


,date_count,customer_count
InvoiceNo,,
536591,2,1
540185,2,1
541596,2,1
541631,2,1
541809,2,1
541816,2,1
541849,2,1
542217,2,1
542806,2,1


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2421,536591,21427,SKULLS STORAGE BOX SMALL,1,2010-12-01 16:57:00,2.10,14606,United Kingdom
2443,536591,51014C,"FEATHER PEN,COAL BLACK",12,2010-12-01 16:57:00,0.85,14606,United Kingdom
2444,536591,21398,RED POLKADOT COFFEE MUG,1,2010-12-01 16:57:00,2.10,14606,United Kingdom
2445,536591,22476,EMPIRE UNION JACK TV DINNER TRAY,2,2010-12-01 16:57:00,4.95,14606,United Kingdom
2446,536591,21643,ASSORTED TUTTI FRUTTI MIRROR,3,2010-12-01 16:57:00,1.25,14606,United Kingdom
...,...,...,...,...,...,...,...,...
498951,578548,84836,ZINC METAL HEART DECORATION,5,2011-11-24 15:02:00,1.25,17345,United Kingdom
498992,578548,22411,JUMBO SHOPPER VINTAGE RED PAISLEY,1,2011-11-24 15:03:00,2.08,17345,United Kingdom
108917,C545569,22840,ROUND CAKE TIN VINTAGE RED,-1,2011-03-03 16:26:00,7.95,14239,United Kingdom
108916,C545569,22423,REGENCY CAKESTAND 3 TIER,-3,2011-03-03 16:26:00,12.75,14239,United Kingdom


,min_date,max_date,time_diff,unique_dates,customer_ids
InvoiceNo,,,,,
536591,2010-12-01 16:57:00,2010-12-01 16:58:00,0 days 00:01:00,2,[14606]
554116,2011-05-22 15:11:00,2011-05-22 15:12:00,0 days 00:01:00,2,[14514]
550320,2011-04-17 12:37:00,2011-04-17 12:38:00,0 days 00:01:00,2,[12748]
550333,2011-04-17 14:05:00,2011-04-17 14:06:00,0 days 00:01:00,2,[15410]
550641,2011-04-19 15:53:00,2011-04-19 15:54:00,0 days 00:01:00,2,[]
550645,2011-04-19 16:30:00,2011-04-19 16:31:00,0 days 00:01:00,2,[16098]
552000,2011-05-05 15:55:00,2011-05-05 15:56:00,0 days 00:01:00,2,[]
553199,2011-05-15 15:13:00,2011-05-15 15:14:00,0 days 00:01:00,2,[17758]
553375,2011-05-16 14:52:00,2011-05-16 14:53:00,0 days 00:01:00,2,[14911]


In [8]:
print("Stockcode",df['StockCode'].unique().tolist())
print("Desc",df['Description'].unique().tolist())

Stockcode ['85123A', 71053, '84406B', '84029G', '84029E', 22752, 21730, 22633, 22632, 84879, 22745, 22748, 22749, 22310, 84969, 22623, 22622, 21754, 21755, 21777, 48187, 22960, 22913, 22912, 22914, 21756, 22728, 22727, 22726, 21724, 21883, 10002, 21791, 21035, 22326, 22629, 22659, 22631, 22661, 21731, 22900, 21913, 22540, 22544, 22492, 'POST', 22086, 20679, 37370, 21871, 21071, 21068, 82483, 82486, 82482, '82494L', 21258, 22114, 21733, 22386, '85099C', 21033, 20723, '84997B', '84997C', 21094, 20725, 21559, 22352, 21212, 21975, 21977, 84991, '84519A', '85183B', '85071B', 21931, 21929, 22961, 22139, 84854, 22411, 82567, 21672, 22774, 22771, 71270, 22262, 22637, 21934, 21169, 21166, 21175, '37444A', '37444C', 22083, '84971S', 47580, 22261, 84832, 22644, 21533, 21557, '15056BL', '15056N', 22646, 22176, 22438, 22778, 22719, 21523, 'D', 21912, 21832, 22379, 22381, 22798, 22926, 22839, 22838, 22783, '35004C', 82484, 84755, 22464, 21324, 22457, 22469, 22470, 22224, 21340, 22189, 22427, 22428, 

In [9]:
summary_df = df.groupby('StockCode').agg(
    total_records=('Description', 'count'),
    unique_desc_count=('Description', 'nunique'),
    descriptions=('Description', lambda x: list(x.dropna().unique()))
).reset_index()

for _, row in summary_df.iterrows():
    print(row['StockCode'], ":", row['descriptions'])

10002 : ['INFLATABLE POLITICAL GLOBE ']
10080 : ['GROOVY CACTUS INFLATABLE', 'check']
10120 : ['DOGGY RUBBER']
10125 : ['MINI FUNKY DESIGN TAPES']
10133 : ['COLOURING PENCILS BROWN TUBE', 'damaged']
10134 : []
10135 : ['COLOURING PENCILS BROWN TUBE']
11001 : ['ASSTD DESIGN RACING CAR PEN']
15030 : ['FAN BLACK FRAME ']
15034 : ['PAPER POCKET TRAVELING FAN ']
15036 : ['ASSORTED COLOURS SILK FAN']
15039 : ['SANDALWOOD FAN']
16008 : ['SMALL FOLDING SCISSOR(POINTED EDGE)', 'check']
16010 : ['FOLDING CAMPING SCISSOR W/KNIF & S']
16011 : ['ANIMAL STICKERS']
16012 : ['FOOD/DRINK SPONGE STICKERS']
16014 : ['SMALL CHINESE STYLE SCISSOR']
16015 : ['MEDIUM CHINESE STYLE SCISSOR']
16016 : ['LARGE CHINESE STYLE SCISSOR']
16033 : ['MINI HIGHLIGHTER PENS']
16043 : ['POP ART PUSH DOWN RUBBER ']
16045 : ['POPART WOODEN PENCILS ASST', 'check']
16046 : ['TEATIME PEN CASE & PENS']
16048 : ['TEATIME ROUND PENCIL SHARPENER ']
16049 : ['TEATIME GEL PENS ASST']
16052 : ['TEATIME PUSH DOWN RUBBER']
16053 : []
1

## Berdasarkan Seluruh Test Di Atas

Suatu transaksi akan memiliki `InvoiceCode` berbeda. Namun, ditemukan bahwa pada 30 transaksi ada beberapa item yang memiliki `InvoiceCode` sama tetapi `InvoiceDate` berbeda sekitar 1 menit.
Dua atau lebih `StockCode` dapat mewaliki `Description` yang sama, bahkan Suatu `StockCode` dapat mewakili beberapa `Description` disaat yang sama.


# Preprocessing

In [10]:
# Simpan ukuran awal sebagai dasar perbandingan sebelum/sesudah pembersihan
rows_before = len(df)
invoices_before = df["InvoiceNo"].nunique()

# Buang deskripsi kosong, invoice pembatalan (prefix C), serta kuantitas nol/negatif
df = df.dropna(subset=["Description"]).copy()
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")].copy()
df = df[df["Quantity"] > 0].copy()

rows_after = len(df)
invoices_after = df["InvoiceNo"].nunique()
print(f"Sebelum pembersihan: {rows_before:,} baris; {invoices_before:,} invoice")
print(f"Sesudah pembersihan: {rows_after:,} baris; {invoices_after:,} invoice")


Sebelum pembersihan: 541,909 baris; 25,900 invoice
Sesudah pembersihan: 530,693 baris; 20,136 invoice


In [11]:
# Setiap invoice menjadi transaksi; drop_duplicates menjaga satu produk dihitung sekali per invoice
df_transaction = (
    df.groupby("InvoiceNo")["Description"]
    .apply(lambda items: list(pd.unique(items)))
    .reset_index(name="Transactions")
)

print("Jumlah transaksi:", len(df_transaction))
display(df_transaction.head())


Jumlah transaksi: 20136


,InvoiceNo,Transactions
0,536365,"[WHITE HANGING HEART T-LIGHT HOLDER, WHITE MET..."
1,536366,"[HAND WARMER UNION JACK, HAND WARMER RED POLKA..."
2,536367,"[ASSORTED COLOUR BIRD ORNAMENT, POPPY'S PLAYHO..."
3,536368,"[JAM MAKING SET WITH JARS, RED COAT RACK PARIS..."
4,536369,[BATH BUILDING BLOCK WORD]


In [12]:
# 1. Inisialisasi TransactionEncoder dari mlxtend
te = TransactionEncoder()

# 2. Transformasi list transaksi menjadi array boolean
te_ary = te.fit(df_transaction["Transactions"]).transform(
    df_transaction["Transactions"]
)

# 3. Konversi array ke DataFrame dan ubah ke biner (1/0)
basket_df = pd.DataFrame(te_ary, columns=te.columns_).astype(int)

# 4. (Opsional) Tetapkan InvoiceNo sebagai index agar identitas transaksi tetap ada
basket_df.index = df_transaction["InvoiceNo"]

display(basket_df.head())

,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,returned,taig adjust,test,to push order througha s stock was,website fixed,wrongly coded 20713,wrongly coded 23343,wrongly marked,wrongly marked 23343,wrongly sold (22719) barcode
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
display(basket_df.shape)
display(df.drop_duplicates(subset=['InvoiceNo']).shape)

(20136, 4077)

(20136, 8)

# Modeling

In [14]:
# Ambang awal sesuai instruksi praktikum
MIN_SUPPORT = 0.01
MIN_CONFIDENCE = 0.5

frequent_itemsets_fpgrowth = fpgrowth(
    basket_df,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=3
)
print("Jumlah frequent itemset:", len(frequent_itemsets_fpgrowth))
display(frequent_itemsets_fpgrowth.sort_values("support", ascending=False).head(10))


/media/dff/Elements1/ITS_semester3/Datamining/.venv/lib/python3.12/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Jumlah frequent itemset: 1837


,support,itemsets
0,0.112237,frozenset({WHITE HANGING HEART T-LIGHT HOLDER})
92,0.103894,frozenset({JUMBO BAG RED RETROSPOT})
297,0.098778,frozenset({REGENCY CAKESTAND 3 TIER})
587,0.083731,frozenset({PARTY BUNTING})
42,0.077672,frozenset({LUNCH BAG RED RETROSPOT})
7,0.072259,frozenset({ASSORTED COLOUR BIRD ORNAMENT})
617,0.068782,frozenset({SET OF 3 CAKE TINS PANTRY DESIGN })
43,0.065554,frozenset({PACK OF 72 RETROSPOT CAKE CASES})
174,0.063220,frozenset({LUNCH BAG BLACK SKULL.})
79,0.062028,frozenset({NATURAL SLATE HEART CHALKBOARD })


In [15]:
# Aturan disaring pada confidence minimum 0.5, lalu diurutkan berdasarkan lift
rules = association_rules(
    frequent_itemsets_fpgrowth,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)
rules_sorted = rules.sort_values(by="lift", ascending=False).reset_index(drop=True)

rules_display = rules_sorted[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
rules_display["antecedents"] = rules_display["antecedents"].apply(lambda x: ", ".join(sorted(x)))
rules_display["consequents"] = rules_display["consequents"].apply(lambda x: ", ".join(sorted(x)))
print(f"Jumlah aturan (support ≥ {MIN_SUPPORT:.2f}, confidence ≥ {MIN_CONFIDENCE:.2f}): {len(rules_display)}")
display(rules_display)


Jumlah aturan (support ≥ 0.01, confidence ≥ 0.50): 727


,antecedents,consequents,support,confidence,lift
0,HERB MARKER THYME,"HERB MARKER PARSLEY, HERB MARKER ROSEMARY",0.010081,0.856540,80.219959
1,"HERB MARKER PARSLEY, HERB MARKER ROSEMARY",HERB MARKER THYME,0.010081,0.944186,80.219959
2,HERB MARKER ROSEMARY,"HERB MARKER PARSLEY, HERB MARKER THYME",0.010081,0.845833,79.961033
3,"HERB MARKER PARSLEY, HERB MARKER THYME",HERB MARKER ROSEMARY,0.010081,0.953052,79.961033
4,HERB MARKER ROSEMARY,HERB MARKER THYME,0.010975,0.920833,78.235865
...,...,...,...,...,...
722,SUKI SHOULDER BAG,JUMBO BAG RED RETROSPOT,0.014005,0.532075,5.121354
723,GREEN REGENCY TEACUP AND SAUCER,REGENCY CAKESTAND 3 TIER,0.025278,0.501478,5.076801
724,TOY TIDY PINK POLKADOT,JUMBO BAG RED RETROSPOT,0.012118,0.520256,5.007587
725,RED RETROSPOT SHOPPER BAG,JUMBO BAG RED RETROSPOT,0.014303,0.500870,4.820989


In [16]:
# Tiga aturan dengan lift tertinggi dan interpretasi ringkas untuk konteks ritel
top_3_rules = rules_display.head(3).copy()
display(top_3_rules)

if top_3_rules.empty:
    print("Belum ada aturan yang memenuhi ambang. Pertimbangkan menurunkan support/confidence dan jelaskan alasannya.")
else:
    for rank, row in enumerate(top_3_rules.itertuples(index=False), start=1):
        print(
            f"{rank}. Jika pelanggan membeli [{row.antecedents}], pembelian [{row.consequents}] "
            f"terjadi bersama dengan support {row.support:.3f}, confidence {row.confidence:.3f}, "
            f"dan lift {row.lift:.3f}. Lift > 1 menunjukkan asosiasi positif; "
            "ritel dapat mempertimbangkan penempatan berdekatan atau promosi bundel."
        )


,antecedents,consequents,support,confidence,lift
0,HERB MARKER THYME,"HERB MARKER PARSLEY, HERB MARKER ROSEMARY",0.010081,0.856540,80.219959
1,"HERB MARKER PARSLEY, HERB MARKER ROSEMARY",HERB MARKER THYME,0.010081,0.944186,80.219959
2,HERB MARKER ROSEMARY,"HERB MARKER PARSLEY, HERB MARKER THYME",0.010081,0.845833,79.961033


1. Jika pelanggan membeli [HERB MARKER THYME], pembelian [HERB MARKER PARSLEY, HERB MARKER ROSEMARY] terjadi bersama dengan support 0.010, confidence 0.857, dan lift 80.220. Lift > 1 menunjukkan asosiasi positif; ritel dapat mempertimbangkan penempatan berdekatan atau promosi bundel.
2. Jika pelanggan membeli [HERB MARKER PARSLEY, HERB MARKER ROSEMARY], pembelian [HERB MARKER THYME] terjadi bersama dengan support 0.010, confidence 0.944, dan lift 80.220. Lift > 1 menunjukkan asosiasi positif; ritel dapat mempertimbangkan penempatan berdekatan atau promosi bundel.
3. Jika pelanggan membeli [HERB MARKER ROSEMARY], pembelian [HERB MARKER PARSLEY, HERB MARKER THYME] terjadi bersama dengan support 0.010, confidence 0.846, dan lift 79.961. Lift > 1 menunjukkan asosiasi positif; ritel dapat mempertimbangkan penempatan berdekatan atau promosi bundel.
